# 06 · Quantization with bitsandbytes

This notebook uses the real **bitsandbytes** library to store weights in 8-bit and 4-bit. You will quantise a single layer and measure the error, compare the NF4 and FP4 formats, load a real LLM in fp32, 8-bit and 4-bit, and compare memory, perplexity and generated text. It closes with when to use bitsandbytes and when to use GPTQ, AWQ or GGUF.

**Time:** ~30 min · **Runs:** offline on CPU in about 1.5 min · **Needs:** [02 · Tokenizers, models and generate](02_tokenizers_models_and_generate.ipynb); the maths from scratch is in [quantization int8/int4/nf4](../07_genai_foundations/07_quantization_int8_int4_nf4.ipynb)

## Why this matters in interviews

- "How would you serve a 70B model on one GPU?" and "LoRA vs QLoRA?" both come down to quantization, and the interviewer wants numbers: bytes per parameter, the quality cost, and what speeds up and what doesn't.
- bitsandbytes is the library behind `load_in_8bit` / `load_in_4bit` in transformers and behind QLoRA. You should know its two modes (LLM.int8 and NF4) and what `bnb_4bit_compute_dtype` and `double_quant` do.
- The strongest answers ask "quantised *what*?" and "on which hardware?". Weight-only quantization cuts memory, but it only speeds things up when there are kernels built for it.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `in07` | What is quantisation and which method would you pick? |
| `fo30` | What is quantisation, in one paragraph, and what does it cost you? |
| `tn05` | LoRA vs QLoRA — what exactly is the difference? |
| `in30` | How would you serve a 70B model on a single 80 GB GPU? |
| `tr33` | What is the difference between weight quantisation and KV cache quantisation? |
| `tr42` | The memory footprint of this 3B model is 12.9 GB. Why, and how would you cut it? |
| `in35` | Explain the arithmetic intensity argument for why decode is memory-bound. |

In [ ]:
import gc, time, logging, warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import bitsandbytes as bnb
import bitsandbytes.functional as bnbF
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

transformers.utils.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)   # hides "inputs will be cast to float16" notices
warnings.filterwarnings("ignore")
torch.manual_seed(0)
print(f"bitsandbytes {bnb.__version__} | transformers {transformers.__version__} | torch {torch.__version__} "
      f"| CUDA available: {torch.cuda.is_available()}")

## 1. Why quantise: memory is parameters × bytes

**In plain English:** a model is billions of numbers. Storing each one in 1 byte, or half a byte, instead of 2 or 4 makes the model 2–8× smaller. It is like saving a photo as a JPEG: much smaller, slightly lossy.

| Format | Bytes/param | 8 B model | 70 B model | Typical use |
|---|---|---|---|---|
| fp32 | 4 | 32 GB | 280 GB | CPU default in PyTorch; master weights in training |
| bf16 / fp16 | 2 | 16 GB | 140 GB | how models are released and served |
| int8 (LLM.int8) | 1 | 8 GB | 70 GB | near-lossless memory saving |
| 4-bit (NF4, GPTQ, AWQ) | 0.5 (+ a little for scales) | 4–5 GB | 35–40 GB | fitting big models on small GPUs; QLoRA |

Memory matters in two ways. It decides whether a model **fits** at all. And decoding is **memory-bandwidth-bound** (`in35`): generating each token means reading every weight once, so fewer bytes per weight means faster tokens, *if* a kernel can compute directly on the compressed weights.

Always ask "quantised what?". This notebook quantises **weights**. Activation quantization (int8/fp8 matmuls) and **KV cache** quantization are separate levers (`tr33`). KV cache quantization buys batch size and context length, not model size.

## 2. One layer in 8-bit and 4-bit

bitsandbytes provides drop-in replacements for `nn.Linear`. The weights are quantised when the layer is moved to a device (`.to("cpu")` or `.cuda()`):

| Layer | Stores | Computes |
|---|---|---|
| `bnb.nn.Linear8bitLt` | int8 weights + one fp scale per output row (`SCB`) | int8 matmul, with outlier features kept in fp16 (LLM.int8) |
| `bnb.nn.Linear4bit` | 4-bit codes packed 2 per byte + one absmax per block of 64 weights | dequantise block by block, then matmul in `compute_dtype` |

In [ ]:
def rel_err(a, b):
    return ((a.float() - b.float()).norm() / b.float().norm()).item()

fp_layer = torch.nn.Linear(1024, 1024)
x = torch.randn(16, 1024)
with torch.no_grad():
    y_ref = fp_layer(x)

int8_layer = bnb.nn.Linear8bitLt(1024, 1024, has_fp16_weights=False, threshold=6.0)
int8_layer.load_state_dict(fp_layer.state_dict())
int8_layer = int8_layer.to("cpu").eval()                 # quantisation happens on the move to a device

layers = {"fp32 nn.Linear": fp_layer, "Linear8bitLt (int8)": int8_layer}
for qt in ["nf4", "fp4"]:
    l4 = bnb.nn.Linear4bit(1024, 1024, compute_dtype=torch.float32, quant_type=qt, compress_statistics=False)
    l4.load_state_dict(fp_layer.state_dict())
    layers[f"Linear4bit ({qt})"] = l4.to("cpu").eval()

def weight_bytes(layer):
    w = layer.weight
    extra = 0
    if hasattr(w, "SCB") and w.SCB is not None:          # int8: per-row scales
        extra = w.SCB.numel() * w.SCB.element_size()
    if getattr(w, "quant_state", None) is not None:       # 4-bit: per-block absmax
        extra = w.quant_state.absmax.numel() * w.quant_state.absmax.element_size()
    return w.numel() * w.element_size() + extra

results = {}
with torch.no_grad():
    for name, layer in layers.items():
        results[name] = (weight_bytes(layer), rel_err(layer(x), y_ref))
        print(f"{name:22s} weight dtype {str(layer.weight.dtype):13s} {results[name][0] / 1e6:5.2f} MB   "
              f"output error vs fp32: {results[name][1]:.4f}")

assert results["Linear8bitLt (int8)"][1] < 0.02                  # int8: under 2% error
assert results["Linear4bit (nf4)"][1] < results["Linear4bit (fp4)"][1] < 0.2
assert results["Linear4bit (nf4)"][0] < results["fp32 nn.Linear"][0] / 7

Reading the numbers:

- **int8** stores 1/4 of the fp32 bytes and changes the output by well under 1%, which is essentially free.
- **4-bit** stores about 1/8 of the bytes (the 0.5 MB of packed codes plus 64 KB of block scales) and gives roughly 10% output error on this random layer. Real LLM weights tolerate that better than you might expect (section 4), but the error is not zero.
- **NF4 beats FP4** at the same size. The next section shows why.
- The 4-bit weight's `dtype` is `uint8` and its shape is flattened, because two 4-bit codes are packed into each byte.

## 3. NF4 vs FP4, and the block size

**In plain English:** 4 bits give only 16 possible values. The question is *where* to put those 16 values. Trained weights are roughly bell-shaped (normally distributed), with most values near zero. **NF4** ("normal float 4") places its 16 levels at the quantiles of a normal distribution: dense near zero where most weights are, sparse in the tails. **FP4** is a tiny floating-point format whose levels suit a normal distribution less well.

Each **block** of weights (64 by default) is scaled by its own maximum absolute value (absmax). Smaller blocks follow local ranges more closely, so the error is lower, but they need more scale values to store.

In [ ]:
W = torch.randn(1024, 1024) * 0.02                     # weights look roughly like this: normal, small std
_, st_nf4 = bnbF.quantize_4bit(W, blocksize=64, quant_type="nf4")
_, st_fp4 = bnbF.quantize_4bit(W, blocksize=64, quant_type="fp4")
nf4_levels = st_nf4.code.numpy()
fp4_levels = np.unique(st_fp4.code.numpy())
print("NF4 levels:", nf4_levels.round(3))
print("FP4 levels:", fp4_levels.round(3))

blocks = W.flatten().view(-1, 64)                                          # blocks of 64 weights
normed = (blocks / blocks.abs().max(dim=1, keepdim=True).values).flatten()  # what quantisation sees: [-1, 1]
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.hist(normed.numpy(), bins=80, color="#adb5bd", density=True, label="weights / block absmax")
ax.vlines(nf4_levels, 0, 0.6, color="#3b5bdb", lw=1.5, label="NF4 levels")
ax.vlines(fp4_levels, 0.6, 1.2, color="#e03131", lw=1.5, label="FP4 levels")
ax.set_xlabel("normalised weight value"); ax.set_ylabel("density")
ax.set_title("NF4 puts its 16 levels where the weights are"); ax.legend(fontsize=8, loc="upper right")
plt.show()

In [ ]:
blocksizes = [64, 128, 256, 512, 1024, 4096]
errs = {"nf4": [], "fp4": []}
for qt in errs:
    for bs in blocksizes:
        q, st = bnbF.quantize_4bit(W, blocksize=bs, quant_type=qt)
        errs[qt].append(rel_err(bnbF.dequantize_4bit(q, st), W))
for qt in errs:
    print(qt, "relative reconstruction error by blocksize:", dict(zip(blocksizes, np.round(errs[qt], 4))))

fig, ax = plt.subplots(figsize=(8, 3.5))
for qt, c in [("nf4", "#3b5bdb"), ("fp4", "#e03131")]:
    ax.plot(blocksizes, errs[qt], "o-", color=c, label=qt)
ax.set_xscale("log", base=2); ax.set_xticks(blocksizes, blocksizes)
ax.set_xlabel("block size (weights per absmax scale)"); ax.set_ylabel("relative error ||W - W_q|| / ||W||")
ax.set_title("4-bit reconstruction error: NF4 < FP4, and smaller blocks help"); ax.legend()
plt.show()
assert all(n < f for n, f in zip(errs["nf4"], errs["fp4"]))
assert errs["nf4"][0] < errs["nf4"][-1]

NF4 wins at every block size, and smaller blocks help both formats. Block size 64 is the QLoRA default. Its cost is one fp32 absmax per 64 weights, which is 0.5 extra bits per weight.

**Double quantization** (`bnb_4bit_use_double_quant=True`, or `compress_statistics=True` at this level) quantises those absmax scales themselves to 8 bits, in blocks of 256. That cuts the overhead from 0.5 to about 0.13 bits per weight. The QLoRA paper quotes 0.127 bits, roughly 3 GB saved on a 65 B model; this small matrix pays a little extra for the fixed-size code table.

In [ ]:
q_plain, st_plain = bnbF.quantize_4bit(W, blocksize=64, quant_type="nf4", compress_statistics=False)
q_dq, st_dq = bnbF.quantize_4bit(W, blocksize=64, quant_type="nf4", compress_statistics=True)

def scale_bits_per_weight(st):
    b = st.absmax.numel() * st.absmax.element_size()
    if st.nested:                                   # second-level scales for the quantised absmax
        b += st.state2.absmax.numel() * st.state2.absmax.element_size() + st.state2.code.numel() * 4
    return 8 * b / W.numel()

print(f"scale overhead, plain  : {scale_bits_per_weight(st_plain):.3f} bits/weight (absmax dtype {st_plain.absmax.dtype})")
print(f"scale overhead, double : {scale_bits_per_weight(st_dq):.3f} bits/weight (absmax dtype {st_dq.absmax.dtype})")
err_plain = rel_err(bnbF.dequantize_4bit(q_plain, st_plain), W)
err_dq = rel_err(bnbF.dequantize_4bit(q_dq, st_dq), W)
print(f"reconstruction error   : {err_plain:.4f} plain vs {err_dq:.4f} double-quantised")
assert scale_bits_per_weight(st_dq) < scale_bits_per_weight(st_plain) / 3 and abs(err_dq - err_plain) < 0.002

## 4. A real LLM: fp32 vs 8-bit vs 4-bit

Now load `SmolLM2-135M-Instruct` four ways through transformers' `BitsAndBytesConfig`. Each run records the memory footprint, the **perplexity** on a fixed paragraph (lower is better; it is exp of the average next-token loss, so it measures how surprised the model is by real text), a short greedy reply, and the decode speed.

`dtype=torch.float32` sets the dtype of the parts that are **not** quantised (embeddings, norms, `lm_head`) and `bnb_4bit_compute_dtype` sets the dtype the 4-bit layers compute in. On a GPU you would use bf16 for both. `device_map="cpu"` keeps everything on the CPU.

In [ ]:
MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
paragraph = ("The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. "
             "It is named after the engineer Gustave Eiffel, whose company designed and built the tower "
             "from 1887 to 1889 as the entrance arch to the 1889 World's Fair.")
ppl_ids = tok(paragraph, return_tensors="pt")
chat = tok.apply_chat_template([{"role": "user", "content": "Give me one tip for a job interview."}],
                               add_generation_prompt=True, return_tensors="pt")

variants = {
    "fp32":        dict(dtype=torch.float32),
    "int8":        dict(quantization_config=BitsAndBytesConfig(load_in_8bit=True)),
    "nf4":         dict(quantization_config=BitsAndBytesConfig(
                        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float32)),
    "nf4 + double": dict(quantization_config=BitsAndBytesConfig(
                        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                        bnb_4bit_compute_dtype=torch.float32)),
}
print(f"{len(ppl_ids['input_ids'][0])} tokens in the perplexity paragraph; prompt has {chat['input_ids'].shape[1]} tokens")

In [ ]:
def quant_state_bytes(model):
    """4-bit scales live outside the parameters, so get_memory_footprint() does not count them."""
    total = 0
    for m in model.modules():
        st = getattr(getattr(m, "weight", None), "quant_state", None)
        if st is not None:
            total += st.absmax.numel() * st.absmax.element_size()
            if st.nested:
                total += st.state2.absmax.numel() * st.state2.absmax.element_size()
    return total

def evaluate(name, kwargs, new_tokens=10):
    t0 = time.perf_counter()
    model = AutoModelForCausalLM.from_pretrained(MODEL, device_map="cpu", **({"dtype": torch.float32} | kwargs)).eval()
    load_s = time.perf_counter() - t0
    with torch.no_grad():
        ppl = torch.exp(model(**ppl_ids, labels=ppl_ids["input_ids"]).loss).item()
        t0 = time.perf_counter()
        out = model.generate(**chat, max_new_tokens=new_tokens, do_sample=False)
        gen_s = time.perf_counter() - t0
    reply = tok.decode(out[0, chat["input_ids"].shape[1]:], skip_special_tokens=True)
    q_proj = model.model.layers[0].self_attn.q_proj
    row = dict(mb=model.get_memory_footprint() / 1e6, scales_mb=quant_state_bytes(model) / 1e6, ppl=ppl,
               tok_s=(out.shape[1] - chat["input_ids"].shape[1]) / gen_s, reply=reply,
               layer=type(q_proj).__name__, load_s=load_s)
    del model; gc.collect()
    return row

runs = {}
for name, kwargs in variants.items():
    try:
        runs[name] = evaluate(name, kwargs)
        r = runs[name]
        print(f"{name:13s} {r['layer']:13s} {r['mb']:6.1f} MB (+{r['scales_mb']:.2f} MB scales)  "
              f"ppl {r['ppl']:5.2f}  {r['tok_s']:4.1f} tok/s  load {r['load_s']:.1f}s")
    except Exception as e:      # CPU support depends on the bitsandbytes version: explain, don't crash
        print(f"{name:13s} FAILED on this machine: {type(e).__name__}: {str(e)[:150]}")
        print("              bitsandbytes CPU support arrived in 0.45+; on older versions this path needs a CUDA GPU.")

In [ ]:
for name, r in runs.items():
    print(f"[{name}] {r['reply']!r}")

In [ ]:
names = list(runs)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
colors = ["#868e96", "#4c6ef5", "#2f9e44", "#37b24d"][:len(names)]
axes[0].bar(names, [runs[n]["mb"] for n in names], color=colors)
axes[0].set_ylabel("MB"); axes[0].set_title("get_memory_footprint()")
for i, n in enumerate(names):
    axes[0].text(i, runs[n]["mb"] + 5, f"{runs[n]['mb']:.0f}", ha="center", fontsize=9)
axes[1].bar(names, [runs[n]["ppl"] for n in names], color=colors)
axes[1].set_ylabel("perplexity (lower = better)"); axes[1].set_title("Perplexity on a fixed paragraph")
for i, n in enumerate(names):
    axes[1].text(i, runs[n]["ppl"] + 0.08, f"{runs[n]['ppl']:.2f}", ha="center", fontsize=9)
plt.tight_layout(); plt.show()

if {"fp32", "int8", "nf4"} <= set(runs):
    assert runs["int8"]["mb"] < runs["fp32"]["mb"] / 2 and runs["nf4"]["mb"] < runs["int8"]["mb"]
    assert runs["fp32"]["ppl"] < runs["int8"]["ppl"] < runs["nf4"]["ppl"]      # less memory, more error
    assert runs["int8"]["ppl"] / runs["fp32"]["ppl"] < 1.15                     # int8 is nearly free

What the experiment shows (`fo30`, `tr42`):

- **Memory:** int8 is well under half of fp32, and NF4 is smaller again. The savings are smaller than 4× and 8× because **the embedding matrix, tied to `lm_head`, is not quantised**. bitsandbytes only replaces `nn.Linear` layers, and here the 28 M-parameter embedding is a fifth of the model. In a 7 B model it is a few percent, and 4-bit gets close to the full 8× saving over fp32 (4× over bf16).
- **Quality:** int8 barely moves perplexity. NF4 costs noticeably more, and in a model this small every weight matters. Larger models lose relatively less at 4-bit, which is why "4-bit for 70 B" is standard advice while 4-bit for a 135 M model is a demo. Measure it on *your* eval before you ship it.
- **Double quant** shrinks the 4-bit scales about 4× (the `+ MB scales` column) and leaves perplexity almost unchanged. On this model that is a sliver next to the unquantised embedding; at 30 B+ scale it is gigabytes.
- **`get_memory_footprint()` does not count the 4-bit scales** (they live in `quant_state`, not in the parameters), so the helper adds them separately.
- **Speed:** on this CPU the quantised models are **not faster**, and int8 is the slowest. The CPU backend dequantises on the fly and has no fast int8 matmul with outlier handling. The fast fused kernels are CUDA ones. So on CPU, bitsandbytes buys *memory*; for fast CPU inference use GGUF with llama.cpp or Ollama (table below).

## 5. LLM.int8(): the outlier threshold

**In plain English:** a few hidden-state features in large transformers are huge (often 20–100× the rest). If you squeeze a row containing one of those into int8, the scale is set by the giant, so all the small values round to zero. **LLM.int8()** splits the matmul in two: feature columns with any value above `llm_int8_threshold` (default **6.0**) are computed in fp16, and the rest (about 99.9% of them) in int8. The demo gives a layer two outlier features:

In [ ]:
x_out = torch.randn(16, 1024)
x_out[:, 7] *= 40.0
x_out[:, 300] *= 30.0                      # two "emergent outlier" features, like the ones LLMs develop
with torch.no_grad():
    y_out = fp_layer(x_out)
    errs_t = {}
    for threshold in [0.0, 6.0]:
        layer = bnb.nn.Linear8bitLt(1024, 1024, has_fp16_weights=False, threshold=threshold)
        layer.load_state_dict(fp_layer.state_dict())
        layer = layer.to("cpu").eval()
        errs_t[threshold] = (rel_err(layer(x), y_ref), rel_err(layer(x_out), y_out))
        print(f"threshold {threshold}: error on normal input {errs_t[threshold][0]:.4f} | "
              f"on input with outliers {errs_t[threshold][1]:.4f}")
assert errs_t[6.0][1] < errs_t[0.0][1] / 3          # the fp16 side path rescues the outlier case

With `threshold=0` (pure int8), the outliers ruin the precision of everything else in their rows, and the error grows about 10×. With the default 6.0 the outlier columns take the fp16 path and the error goes back to the no-outlier level. Set it with `BitsAndBytesConfig(load_in_8bit=True, llm_int8_threshold=6.0)`. Lowering it sends more columns through fp16 (more accurate, slower); raising it towards infinity makes everything pure int8.

## 6. The GPU recipe (what you'd run for real)

On a CUDA GPU the same config loads a 7–8 B model in about 5 GB of VRAM, which fits a free Colab T4. This is also exactly how QLoRA prepares its frozen base model ([QLoRA with bitsandbytes and PEFT](../11_finetuning/05_qlora_with_bitsandbytes_and_peft.ipynb)). It is not run here, because it needs an NVIDIA GPU and a multi-GB download:

```python
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",              # normal-float 4: best for weights
    bnb_4bit_use_double_quant=True,         # quantise the scales too (~0.4 bit/param saved)
    bnb_4bit_compute_dtype=torch.bfloat16,  # matmuls run in bf16 after on-the-fly dequantisation
)
model_id = "meta-llama/Llama-3.1-8B-Instruct"     # gated: accept the licence, set HF_TOKEN
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
tok = AutoTokenizer.from_pretrained(model_id)
print(f"{model.get_memory_footprint() / 1e9:.1f} GB")   # ~5.5 GB instead of ~16 GB in bf16

# QLoRA next step (see 11_finetuning/05): peft.prepare_model_for_kbit_training(model), then add LoRA adapters.
# The quantised base stays frozen; only the bf16 adapters train.
```

Honest caveats:

- **Hardware:** bitsandbytes was CUDA-only for years. Recent versions (0.45+, and 0.50 here) add CPU, Intel XPU, Apple MPS and other backends of varying completeness. Check the version's release notes for your hardware before you promise it works.
- **Speed:** 4-bit bitsandbytes on a GPU is usually a little *slower* per token than bf16 at batch size 1 (dequantisation overhead), but it fits models that otherwise wouldn't fit. For fast quantised *serving*, use AWQ or GPTQ kernels in vLLM/TGI, or FP8 on H100-class GPUs.
- **Saving:** 4-bit and 8-bit bitsandbytes models can be saved and pushed (`save_pretrained`), but the usual approach is to keep the bf16 checkpoint and quantise at load time.
- **Training:** you cannot fully fine-tune the quantised weights. You train LoRA adapters on top (QLoRA).

## 7. Which quantization method, when

| Method | What it is | Needs calibration data? | Best for | Runs on |
|---|---|---|---|---|
| **bitsandbytes LLM.int8** | 8-bit weights, outliers kept in fp16 | no (quantises at load) | quick near-lossless memory halving | CUDA (plus newer CPU/XPU backends) |
| **bitsandbytes NF4** | 4-bit normal-float weights, blockwise absmax, optional double quant | no | **QLoRA fine-tuning**, trying big models on small GPUs | same |
| **GPTQ** | post-training 4-bit, layer by layer, second-order error correction | yes (a few hundred samples) | fast 4-bit GPU serving (vLLM, TGI, ExLlama) | GPU |
| **AWQ** | activation-aware 4-bit: protects the ~1% of weight channels with large activations | yes (small) | fast 4-bit GPU serving, often slightly better than GPTQ | GPU |
| **GGUF (llama.cpp k-quants)** | a file format with mixed 2–8-bit per-tensor quantization | optional (imatrix) | **CPU and laptop inference**, Ollama, Apple Silicon | CPU, Metal, CUDA |
| **FP8** | 8-bit floating point supported by the hardware | usually no | high-throughput serving on H100/MI300 with real speed-ups | recent GPUs |

A rule of thumb for the interview (`in07`): **fine-tuning → bitsandbytes NF4 (QLoRA); GPU serving → AWQ/GPTQ (or FP8 on new GPUs); CPU/laptop → GGUF**. For `in30`: a 70 B model in bf16 is about 140 GB, and 4-bit AWQ/GPTQ brings it to about 35–40 GB, leaving about 40 GB of an 80 GB card for the KV cache.

## Try it yourself

**1. Bits per weight.** Using `quantize_4bit`, compute the *total* storage in bits per weight (packed codes + scales) for NF4 with block size 64 and with 256, both without double quant. Which is smaller, and by how much does the error grow?

In [ ]:
# Solution — try it yourself first, then run this
for bs in [64, 256]:
    q, st = bnbF.quantize_4bit(W, blocksize=bs, quant_type="nf4")
    bits = 8 * (q.numel() * q.element_size() + st.absmax.numel() * st.absmax.element_size()) / W.numel()
    print(f"blocksize {bs:3d}: {bits:.3f} bits/weight, error {rel_err(bnbF.dequantize_4bit(q, st), W):.4f}")
# 64 -> 4.5 bits, 256 -> 4.125 bits; the larger block saves ~8% of storage for a few % more error.

**2. Quantisation cost vs weight distribution.** Real weights are bell-shaped, but some tensors are closer to uniform. Quantise a *uniform* random matrix (`torch.rand(1024, 1024) * 2 - 1`) with NF4 and FP4 at block size 64. Does NF4 still win?

In [ ]:
# Solution — try it yourself first, then run this
U = torch.rand(1024, 1024) * 2 - 1
for qt in ["nf4", "fp4"]:
    q, st = bnbF.quantize_4bit(U, blocksize=64, quant_type=qt)
    print(f"{qt}: error on uniform weights {rel_err(bnbF.dequantize_4bit(q, st), U):.4f}")

def nf4_over_fp4(M):
    e = {}
    for qt in ["nf4", "fp4"]:
        q, st = bnbF.quantize_4bit(M, blocksize=64, quant_type=qt)
        e[qt] = rel_err(bnbF.dequantize_4bit(q, st), M)
    return e["nf4"] / e["fp4"]
print(f"NF4 error as a fraction of FP4 error: normal {nf4_over_fp4(W):.2f}, uniform {nf4_over_fp4(U):.2f}")
assert nf4_over_fp4(W) < nf4_over_fp4(U) < 1
# NF4 still wins (FP4 wastes two of its 16 codes on +-0.005, right next to zero), but its margin
# shrinks when the values are not bell-shaped: the advantage comes from matching the distribution.

**3. Memory planner.** Write `weights_gb(params_billion, fmt)` for `fmt` in `{"bf16", "int8", "nf4"}`, counting NF4 as 4.127 bits (with double quant, the QLoRA paper's figure). Which formats let a 70 B model's weights fit in an 80 GB GPU with at least 30 GB left for the KV cache?

In [ ]:
# Solution — try it yourself first, then run this
BITS = {"bf16": 16, "int8": 8, "nf4": 4.127}
def weights_gb(params_billion, fmt):
    return params_billion * 1e9 * BITS[fmt] / 8 / 1e9
for fmt in BITS:
    gb = weights_gb(70, fmt)
    print(f"70B in {fmt:4s}: {gb:6.1f} GB weights -> {80 - gb:6.1f} GB left on an 80 GB GPU")
assert [f for f in BITS if 80 - weights_gb(70, f) >= 30] == ["nf4"]

## Say it in an interview

- **30-second answer:** "Quantization stores weights in fewer bits: int8 halves bf16 memory almost for free, 4-bit quarters it with a small, measurable quality cost. bitsandbytes gives you LLM.int8, which keeps outlier features in fp16 above a threshold of 6, and NF4, a 4-bit format whose levels sit at normal-distribution quantiles, with a scale per 64 weights and optional double quantization of the scales. NF4 plus LoRA adapters is QLoRA."
- **Numbers:** bytes per parameter 4 / 2 / 1 / 0.5. 70 B: 140 GB bf16 → 35–40 GB at 4-bit. Scale overhead is 0.5 bit/weight at block 64, about 0.127 with double quant. int8 perplexity is within a few percent; 4-bit costs more on small models.
- **Trade-offs:** memory vs quality vs speed. Weight-only quantization speeds up decode only with kernels that read the compressed weights (decode is memory-bound). On CPU, bitsandbytes saves memory but is slower; use GGUF there. For production GPU serving, AWQ/GPTQ or FP8.
- **Traps:** "4-bit makes it 8× smaller" (not when embeddings and `lm_head` stay in full precision). "Quantised = faster" (only with the right kernels and hardware). "QLoRA is a different algorithm" (no, it's LoRA on an NF4-frozen base). And forgetting the KV cache, which is a separate quantization knob (`tr33`).
- **Follow-ups:** "why NF4 and not int4?" (information-theoretically matched to normal weights), "what is `compute_dtype`?" (the dtype the de-quantised matmul runs in: bf16 on a GPU), "how do you check the quality loss?" (perplexity plus your task eval, compared with bf16, before shipping).

## Next

- The maths from scratch: [quantization int8/int4/nf4](../07_genai_foundations/07_quantization_int8_int4_nf4.ipynb)
- Fine-tune on a quantised base: [QLoRA with bitsandbytes and PEFT](../11_finetuning/05_qlora_with_bitsandbytes_and_peft.ipynb) · [LoRA with PEFT](../11_finetuning/04_lora_with_peft.ipynb)
- Serving speed: [KV cache and inference speed](../07_genai_foundations/06_kv_cache_and_inference_speed.ipynb) · [Local LLMs with Ollama (GGUF)](../08_llm_apps/05_local_llms_with_ollama.ipynb) · [latency and cost](../16_production/05_latency_percentiles_p50_p95_and_cost.ipynb)
- Theory: [genai_flow course](../../genai_flow/index.html), [interview bank](../../ai_interview_prep/index.html)